# 14 — Advanced Autograd: Custom Functions, Higher-Order Grads, Checkpointing, and Stability

Goal: understand autograd internals and advanced differentiation workflows.

_Generated: 2026-01-25_

## Setup

```bash
pip install torch torchvision torchaudio
pip install transformers datasets tokenizers accelerate evaluate
pip install matplotlib tensorboard
pip install safetensors sentencepiece
```

In [ ]:

import os, math, random
import numpy as np
import torch

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("torch:", torch.__version__)
print("device:", device)

## 1. Custom autograd function

Use `torch.autograd.Function` when:
- you need a custom forward that PyTorch cannot differentiate automatically
- you want a custom backward for performance or stability
- you need to wrap external libraries

Rules:
- save tensors for backward using `ctx.save_for_backward`
- backward must return grads for each input

In [ ]:

import torch

class SwishFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig = torch.sigmoid(x)
        ctx.save_for_backward(sig)
        return x * sig
    @staticmethod
    def backward(ctx, grad_out):
        (sig,) = ctx.saved_tensors
        # d/dx (x*sigmoid(x)) = sig + x*sig*(1-sig)
        # grad = grad_out * (sig + x*sig*(1-sig))
        # we need x, but we didn't save x; reconstruct is not possible => save x too in real impl
        # For demonstration: save x as well
        raise RuntimeError("Demonstration: see improved version below.")

class SwishFn2(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        sig = torch.sigmoid(x)
        ctx.save_for_backward(x, sig)
        return x * sig
    @staticmethod
    def backward(ctx, grad_out):
        x, sig = ctx.saved_tensors
        grad = sig + x * sig * (1 - sig)
        return grad_out * grad

def swish(x):
    return SwishFn2.apply(x)

x = torch.randn(5, requires_grad=True)
y = swish(x).sum()
y.backward()
x.grad

## 2. Higher-order gradients

Some applications:
- meta-learning
- hyperparameter optimization
- curvature estimation

Use `create_graph=True` to keep graph for further differentiation.

In [ ]:

import torch
x = torch.randn(3, requires_grad=True)
y = (x**3).sum()
g1 = torch.autograd.grad(y, x, create_graph=True)[0]
g2 = torch.autograd.grad(g1.sum(), x)[0]  # second derivative
g1, g2

## 3. Gradient checkpointing (activation checkpointing)

Checkpointing trades compute for memory by recomputing forward activations in backward.
Useful for large Transformers and deep nets.

API: `torch.utils.checkpoint.checkpoint(function, *args)`

In [ ]:

import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

class DeepBlock(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.l1 = nn.Linear(d, d)
        self.l2 = nn.Linear(d, d)
    def forward(self, x):
        return torch.relu(self.l2(torch.relu(self.l1(x))))

block = DeepBlock(128)

def run_with_ckpt(x):
    return checkpoint(block, x)

x = torch.randn(32, 128, requires_grad=True)
y = run_with_ckpt(x).sum()
y.backward()
x.grad.norm()

## 4. Stability toolkit (advanced)

- gradient clipping
- loss scaling (AMP)
- anomaly detection for debugging
- clamp/softplus alternatives for constrained params
- log-space computations for probabilities